In [84]:
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency, chisquare

# Параметри для Варіанта 1 (Київ)
np.random.seed(42)
base_temp = 9.5
amplitude = 14
city = "Київ"

def season(month):
    if month in (12, 1, 2):
        return "зима"
    if month in (3, 4, 5):
        return "весна"
    if month in (6, 7, 8):
        return "літо"
    return "осінь"

rows = []
for year in [2021, 2022, 2023, 2024]:
    for month in range(1, 13):
        seasonal = amplitude * np.cos((month - 7) / 12 * 2 * np.pi)
        noise = np.random.normal(0, 1.0)
        temp = round(base_temp + seasonal + noise, 1)
        diff = temp - base_temp

        if diff < -3:
            norm_cat = "холодніше"
        elif diff > 3:
            norm_cat = "тепліше"
        else:
            norm_cat = "звичайно"

        rows.append({
            "місто": city,
            "рік": year,
            "місяць": month,
            "температура": temp,
            "сезон": season(month),
            "відхилення_від_норми": norm_cat,
        })

climate = pd.DataFrame(rows)
print("Перші 5 рядків згенерованого набору:")
display(climate.head())

Перші 5 рядків згенерованого набору:


,місто,рік,місяць,температура,сезон,відхилення_від_норми
0,Київ,2021,1,-4.0,зима,холодніше
1,Київ,2021,2,-2.8,зима,холодніше
2,Київ,2021,3,3.1,весна,холодніше
3,Київ,2021,4,11.0,весна,звичайно
4,Київ,2021,5,16.3,весна,тепліше


In [85]:
# 1. Абсолютні частоти
table = pd.crosstab(climate["сезон"], climate["відхилення_від_норми"])
print("Таблиця спряженості (абсолютні частоти):")
display(table)

# 2. Відносні частки в межах кожного рядка
table_norm = pd.crosstab(climate["сезон"], climate["відхилення_від_норми"], normalize="index")
print("\nТаблиця спряженості (normalize='index'):")
display(table_norm.round(2))

Таблиця спряженості (абсолютні частоти):


відхилення_від_норми,звичайно,тепліше,холодніше
сезон,,,
весна,4,4,4
зима,0,0,12
літо,0,12,0
осінь,4,4,4



Таблиця спряженості (normalize='index'):


відхилення_від_норми,звичайно,тепліше,холодніше
сезон,,,
весна,0.33,0.33,0.33
зима,0.00,0.00,1.00
літо,0.00,1.00,0.00
осінь,0.33,0.33,0.33


In [86]:
chi2, pvalue, dof, expected = chi2_contingency(table)

print(f"Статистика χ² = {chi2:.2f}")
print(f"p-value = {pvalue:.4f}")
print(f"Ступені свободи (dof) = {dof}")

expected_df = pd.DataFrame(expected, index=table.index, columns=table.columns)
print("\nМатриця очікуваних частот (expected):")
display(expected_df.round(1))

Статистика χ² = 38.40
p-value = 0.0000
Ступені свободи (dof) = 6

Матриця очікуваних частот (expected):


відхилення_від_норми,звичайно,тепліше,холодніше
сезон,,,
весна,2.0,5.0,5.0
зима,2.0,5.0,5.0
літо,2.0,5.0,5.0
осінь,2.0,5.0,5.0


In [87]:
print("Перевірка умові (expected >= 5):")
display(expected_df >= 5)

Перевірка умові (expected >= 5):


відхилення_від_норми,звичайно,тепліше,холодніше
сезон,,,
весна,False,True,True
зима,False,True,True
літо,False,True,True
осінь,False,True,True


In [88]:
observed_seasons = climate["сезон"].value_counts().reindex(["зима", "весна", "літо", "осінь"])
n = len(climate)
expected_seasons = [0.25 * n, 0.25 * n, 0.25 * n, 0.25 * n]

res_season = chisquare(f_obs=observed_seasons, f_exp=expected_seasons)
print(f"Критерій узгодженості для сезонів: статистика = {res_season.statistic:.2f}, p-value = {res_season.pvalue:.4f}")

Критерій узгодженості для сезонів: статистика = 0.00, p-value = 1.0000


In [89]:
# Підрахунок фактичних частот категорій відхилення
observed_norms = climate["відхилення_від_норми"].value_counts().reindex(["холодніше", "звичайно", "тепліше"])

# Заявлена пропорція: 25% холодніше, 50% звичайно, 25% тепліше
expected_norm_props = [0.25, 0.50, 0.25]
expected_norm_counts = [p * n for p in expected_norm_props]

res_norm = chisquare(f_obs=observed_norms, f_exp=expected_norm_counts)
print(f"Критерій узгодженості для відхилень: статистика = {res_norm.statistic:.2f}, p-value = {res_norm.pvalue:.4f}")

Критерій узгодженості для відхилень: статистика = 21.33, p-value = 0.0000


### Відповіді на контрольні запитання:

1. **Механіка статистики χ² (чому квадрат відхилення і ділення на очікувану частоту):**
   - Квадрат різниці $(O - E)^2$ використовується для того, щоб додатні й від'ємні відхилення фактичних частот ($O$) від очікуваних ($E$) не взаємознищувалися при сумуванні. Ділення на очікувану частоту стандартизує це відхилення, враховуючи масштаб очікуваних значень (похибка для великих чисел є менш критичною, ніж така ж абсолютна похибка для малих).

2. **Як обчислюється очікувана частота клітинки таблиці спряженості:**
   - Очікувана частота обчислюється як добуток суми відповідного рядка на суму відповідного стовпця, поділений на загальну суму всіх спостережень в таблиці: $E = \frac{R_i \cdot C_j}{N}$. Це витікає з теорії ймовірностей для незалежних подій (добуток крайових ймовірностей на загальну кількість).

3. **Різниця між «p-value велике — немає підстав відхилити H0» і «доведено незалежність»:**
   - Велика величина $p$-value означає лише відсутність достатніх доказів для відхилення нульової гіпотези наявними даними (брак статистичної потужності чи замалий обсяг вибірки). У статистиці «не відхилити $H_0$» не означає математичного доведення її абсолютної істинності.

4. **Що робити, якщо очікувана частота менша за 5:**
   - Малі очікувані частоти (< 5) порушують математичні припущення асимптотичного розподілу хі-квадрат Пірсона, що призводить до спотворення рівня значущості тесту. Проблема вирішується шляхом об'єднання суміжних малочисельних категорій або застосуванням точних тестів (наприклад, точного критерію Фішера).